# R4. Modelo final sobre el dataset sintético

**Lo corre cualquiera:** solo necesita `data/synthetic/synthetic_shifts.csv`, `src/experimentos.py`,
`reproducibility/evaluacion.py` y `reports/experimentos/e1_hiperparametros.json`. No usa datos del terminal.

Es el mismo pipeline del notebook 14: partición 80/10/10 por escala en orden temporal, CV de ventana creciente en
train, Ridge afinado (35 numéricas + 5 categóricas) reentrenado en train + val y evaluado en test, frente a las
líneas base y al plan del terminal, por turno y por escala. Al final, la tabla real vs sintético.

Secciones: 1. Datos · 2. Resultados por turno · 3. Resultados por escala · 4. Real vs sintético

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, "..")
import evaluacion as E

pd.set_option("display.width", 200)
RAIZ = Path("..").resolve()
SALIDA = RAIZ / "reports" / "reproducibilidad"
SALIDA.mkdir(parents=True, exist_ok=True)

## 1. Datos

In [2]:
sint = pd.read_csv(RAIZ / "data" / "synthetic" / "synthetic_shifts.csv", parse_dates=["fecha_turno", "inicio_turno"])
turnos = E.preparar(sint)
print(f"{len(turnos):,} turnos · {turnos.groupby(E.ESCALA).ngroups:,} escalas")
turnos.groupby("particion", observed=True).agg(turnos=("cmph", "size"), escalas=("visita_id", "nunique"),
                                               desde=("inicio_turno", "min"), hasta=("inicio_turno", "max"))

1,984 turnos · 963 escalas


,turnos,escalas,desde,hasta
particion,,,,
train,1586,771,2030-01-01 07:00:00,2032-02-04 23:00:00
val,198,98,2032-02-05 07:00:00,2032-05-14 15:00:00
test,200,94,2032-05-15 23:00:00,2032-08-20 23:00:00


In [3]:
resumen, por_turno, por_escala = E.completo(turnos)
print(f"Ridge final · MAE CV {resumen['mae_cv']:.3f} · val {resumen['mae_val']:.3f} · test {resumen['mae_test']:.3f} · "
      f"sesgo test {resumen['sesgo_test']:+.3f}")

Ridge final · MAE CV 2.071 · val 1.762 · test 1.941 · sesgo test -0.141


## 2. Resultados por turno (test)

In [4]:
por_turno.drop(columns="IC_mejora_bajo").to_csv(SALIDA / "sintetico_test_por_turno.csv")
por_turno.drop(columns="IC_mejora_bajo").round(3)

,MAE,IC_MAE,RMSE,R2,mejora_modelo_%,IC_mejora
predictor,,,,,,
modelo,1.941,"[1.754, 2.131]",2.437,0.319,NaN,
b_reciente30,2.378,,3.000,-0.033,18.367,"[10.5, 25.6]"
b_media_tv,2.336,,2.972,-0.013,16.911,"[8.3, 24.5]"
b_servicio_tv,2.417,,3.070,-0.081,19.698,"[10.9, 27.2]"
b_anterior,2.665,,3.332,-0.274,27.156,"[18.9, 34.6]"
plan_terminal,6.199,,7.609,-5.642,68.683,"[62.8, 73.5]"


## 3. Resultados por escala (test; CMPH = movimientos / horas-grúa)

In [5]:
por_escala.to_csv(SALIDA / "sintetico_test_por_escala.csv")
por_escala.round(3)

,escalas,MAE,MAPE_%,R2
predictor,,,,
modelo,94,1.688,6.165,0.291
b_reciente30,94,2.064,7.448,-0.041
b_media_tv,94,2.015,7.353,-0.000
b_servicio_tv,94,2.041,7.432,-0.078
b_anterior,94,1.588,5.723,0.263
plan_terminal,94,6.015,21.517,-7.624


## 4. Real vs sintético

Los criterios de MAE y de orden se evalúan sobre la media de 10 semillas en R2; aquí se muestran para el
dataset publicado, junto con los de IC.

In [6]:
tabla, criterios = E.comparar(resumen, por_turno)
tabla.to_csv(SALIDA / "real_vs_sintetico.csv")
display(tabla.round(3))
criterios

,real,sintetico,dif_%
mae_cv,2.067,2.071,0.215
mae_escala_b_reciente30,1.932,2.064,6.836
mae_escala_plan,3.847,6.015,56.361
mae_escala_test,1.536,1.688,9.910
mae_test,1.963,1.941,-1.111
mae_test_b_anterior,2.280,2.665,16.881
mae_test_b_reciente30,2.264,2.378,5.033
mae_test_b_servicio_tv,2.171,2.417,11.348
mae_test_plan,3.772,6.199,64.332
mae_val,1.986,1.762,-11.255


MAE cv ±5%                                     True
MAE val ±5%                                   False
MAE test ±5%                                   True
mismo orden de predictores (MAE test)         False
Ridge gana a b_reciente30 con IC 95 % > 0      True
Ridge gana a plan_terminal con IC 95 % > 0     True
Name: cumple, dtype: bool